In [2]:
import os
import numpy as np
import pandas as pd
import mysql.connector
from dotenv import load_dotenv
from statsmodels.tsa.statespace.sarimax import SARIMAX
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Chargement des variables d'environnement depuis le fichier .env du dossier parent
load_dotenv(dotenv_path="../.env")

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
# Connexion à la base de données Cloud Aiven MySQL
connection = mysql.connector.connect(
    host=os.getenv("AIVEN_HOST"),
    port=int(os.getenv("AIVEN_PORT", 14198)),
    user=os.getenv("AIVEN_USER"),
    password=os.getenv("AIVEN_PASSWORD"),
    database="defaultdb"
)

In [ ]:
# 1. Extraction d'un échantillon propre
# Note: On utilise la station 3012 (Saxe-Gambetta) ou 1034 qui contiennent des milliers d'enregistrements.
query = """
SELECT date_enregistrement AS date_enregistrement, velos 
FROM stations_historique 
WHERE id = 3012 
ORDER BY date_enregistrement ASC
"""

In [ ]:
# 2. Chargement dans un DataFrame
df = pd.read_sql(query, connection)
connection.close()

In [ ]:
# Vérification rapide du chargement
print("Colonnes chargées :", list(df.columns))
print("Nombre de lignes brutes :", len(df))
if len(df) > 0:
    print(df.head())

In [ ]:
# 3. Préparation des données (Index temporel + rééchantillonnage dynamique)
if 'date_enregistrement' in df.columns:
    df['date_enregistrement'] = pd.to_datetime(df['date_enregistrement'])
    df.set_index('date_enregistrement', inplace=True)

# Calcul de la durée totale des données chargées en heures
time_span_hours = (df.index.max() - df.index.min()).total_seconds() / 3600

# Choix dynamique de la fréquence de rééchantillonnage pour conserver assez de points (min 48)
if time_span_hours < 8:
    freq = '1T'  # Toutes les minutes
    print(f"Durée des données courte ({time_span_hours:.1f}h). Rééchantillonnage par minute ('1T')")
elif time_span_hours < 36:
    freq = '10T' # Toutes les 10 minutes
    print(f"Durée des données moyenne ({time_span_hours:.1f}h). Rééchantillonnage par 10 minutes ('10T')")
elif time_span_hours < 120:
    freq = '30T' # Toutes les 30 minutes
    print(f"Durée des données ({time_span_hours:.1f}h). Rééchantillonnage par 30 minutes ('30T')")
else:
    freq = 'h'   # Toutes les heures
    print(f"Durée des données longue ({time_span_hours:.1f}h). Rééchantillonnage par heure ('h')")

df = df.resample(freq).mean().ffill()
print(f"Nombre de points rééchantillonnés : {len(df)}")

# 4. Génération des graphiques de corrélation temporelle
# Ajustement dynamique du nombre de lags maximum par rapport aux données existantes
lags = min(48, len(df) // 2 - 1)

if lags >= 5:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))
    plot_acf(df['velos'], ax=ax1, lags=lags, title="ACF - Corrélation temporelle")
    plot_pacf(df['velos'], ax=ax2, lags=lags, title="PACF - Corrélation directe")
    plt.tight_layout()
    plt.show()
else:
    print(f"Pas assez de données pour générer l'ACF/PACF (lags calculés = {lags}, minimum requis = 5).")

In [ ]:
# 5. Division Entraînement / Test (80% / 20%) et Évaluation du modèle
if len(df) >= 20:
    train_size = int(len(df) * 0.8)
    train_df = df.iloc[:train_size]
    test_df = df.iloc[train_size:]
    
    print(f"Données d'entraînement : {len(train_df)} points. Données de test : {len(test_df)} points.")
    
    # Définition des paramètres
    order = (1, 1, 1)
    seasonal_order = (0, 0, 0, 0)
    
    if time_span_hours >= 24:
        if freq == 'h': S = 24
        elif freq == '30T': S = 48
        elif freq == '10T': S = 144
        else: S = 24
        seasonal_order = (1, 1, 1, S)
        print(f"Ajustement d'un modèle SARIMAX{order}x{seasonal_order} sur l'entraînement...")
        model = SARIMAX(train_df['velos'], order=order, seasonal_order=seasonal_order)
    else:
        print(f"Ajustement d'un modèle ARIMA{order} simple (historique court)...")
        model = SARIMAX(train_df['velos'], order=order)
        
    results = model.fit(disp=False)
    
    # Prédiction sur la période de test
    predictions = results.predict(start=test_df.index[0], end=test_df.index[-1])
    
    # Calcul des métriques d'erreur
    mae = mean_absolute_error(test_df['velos'], predictions)
    rmse = np.sqrt(mean_squared_error(test_df['velos'], predictions))
    
    print(f"\n--- Métriques de validation (Test Set) ---")
    print(f"Erreur Moyenne Absolue (MAE) : {mae:.2f} vélos")
    print(f"Erreur Quadratique Moyenne (RMSE) : {rmse:.2f} vélos")
    
    # Graphique de comparaison Test vs Prédictions
    plt.figure(figsize=(12, 6))
    plt.plot(train_df.index, train_df['velos'], label='Historique Entraînement', color='blue')
    plt.plot(test_df.index, test_df['velos'], label='Valeurs Réelles (Test)', color='green')
    plt.plot(test_df.index, predictions, label='Prédictions du Modèle', color='red', linestyle='--')
    plt.title("Évaluation du modèle : Valeurs Réelles vs Prédictions")
    plt.xlabel("Date / Heure")
    plt.ylabel("Nombre de vélos disponibles")
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print(f"Pas assez de données pour diviser en Train/Test (minimum 20 requis, actuel: {len(df)}).")

In [ ]:
# 6. Entraînement final (sur 100% des données) et prévision des prochaines heures (Futur)
if len(df) >= 10:
    print("Entraînement du modèle final sur TOUTES les données disponibles...")
    
    order = (1, 1, 1)
    if time_span_hours < 24:
        model_final = SARIMAX(df['velos'], order=order)
    else:
        # S calculé dynamiquement
        if freq == 'h': S = 24
        elif freq == '30T': S = 48
        elif freq == '10T': S = 144
        else: S = 24
        model_final = SARIMAX(df['velos'], order=order, seasonal_order=(1, 1, 1, S))
        
    results_final = model_final.fit(disp=False)
    
    # Prévoir les 8 prochaines étapes (ex: 8h si fréquence horaire, 40 minutes si fréquence 5-min, etc.)
    forecast_steps = 8
    forecast_output = results_final.get_forecast(steps=forecast_steps)
    
    # Valeurs moyennes prédites et intervalles de confiance à 95%
    forecast_mean = forecast_output.predicted_mean
    confidence_intervals = forecast_output.conf_int(alpha=0.05) # Alpha = 5% -> Confiance à 95%
    
    print(f"\n--- Prévisions pour les {forecast_steps} prochaines périodes ---")
    for i, (idx, val) in enumerate(forecast_mean.items()):
        lower = confidence_intervals.iloc[i, 0]
        upper = confidence_intervals.iloc[i, 1]
        # On borne à 0 et au nombre de places max (ex: 30) pour rester réaliste
        lower = max(0, lower)
        print(f"{idx.strftime('%Y-%m-%d %H:%M')} : {val:.1f} vélos (Intervalle : [{lower:.1f} - {upper:.1f}])")
        
    # Graphique final avec prévisions futures et intervalles de confiance
    plt.figure(figsize=(12, 6))
    # On trace les 100 derniers points historiques pour plus de clarté
    historical_plot_df = df.tail(100)
    plt.plot(historical_plot_df.index, historical_plot_df['velos'], label='Historique Récent', color='blue')
    
    # Prévisions
    plt.plot(forecast_mean.index, forecast_mean, label='Prévision Future', color='orange', marker='o')
    
    # Intervalle de confiance
    plt.fill_between(
        confidence_intervals.index, 
        np.maximum(0, confidence_intervals.iloc[:, 0]), 
        confidence_intervals.iloc[:, 1], 
        color='orange', 
        alpha=0.15, 
        label='Zone de Confiance (95%)'
    )
    
    plt.title("Prévision de la disponibilité des vélos (avec intervalle de confiance)")
    plt.xlabel("Date / Heure")
    plt.ylabel("Vélos disponibles")
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print("Pas assez de données pour l'entraînement final.")